In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install indic-nlp-library

In [3]:
!pip install emoji

In [4]:
!pip install indicsyllabifier

In [5]:
!pip install tokenizers

In [6]:
import pandas as pd

file_path_train = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_CG/train_CG.csv'

# Example: Read a CSV file into a pandas DataFrame
try:
    df_train = pd.read_csv(file_path_train)
    print(f"Successfully loaded df_train - shape: {df_train.shape}", "\n")
    print(df_train.head())

except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Check your path.")

file_path_dev = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_CG/dev_CG.csv'   #test_data_withoutlabelCG.csv
# Example: Read a CSV file into a pandas DataFrame
try:
    df_dev = pd.read_csv(file_path_dev)
    print(f"Successfully loaded df_dev - shape: {df_dev.shape}", "\n")
    print(df_dev.head())

except FileNotFoundError:
    print(f"Error: The file {file_path_dev} was not found. Check your path.")

file_path_test = '/content/drive/MyDrive/Tulu Hope Speech 2026/Hope_CG/test_data.csv'

# Example: Read a CSV file into a pandas DataFrame
try:
    df_test = pd.read_csv(file_path_test)
    print(f"Successfully loaded df_test - shape: {df_test.shape}")
    print(df_test.head())

except FileNotFoundError:
    print(f"Error: The file {file_path_test} was not found. Check your path.")


Successfully loaded df_train - shape: (5991, 2) 

                                                Text              Label
0      Santhu nu masth miss malthondu ullar bro 🤔🤔🤔🤔         uninvolved
1   Undhe n YouTube d thooyere andala mobile bodathe  discouraging hope
2  ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್‌ನ್ ನೀರ್‌ಡ್ ಪಾಡ್ದ್ ಅವು...         uninvolved
3                            Super comedy deepak rai   encouraging hope
4                Namma tulundadu comedy first eppodu         uninvolved
Successfully loaded df_dev - shape: (1284, 2) 

                                                Text             Label
0                      Yeranda narrd korle marre 😂😂😂        uninvolved
1  Tulu barpundge areg. Wow mokedha base tulu. Ja...  encouraging hope
2                          Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale  encouraging hope
3                        bayya poora film padlethe..  encouraging hope
4  Super act super comedy deepak jp thumbinadu ra...  encouraging hope
Successfully loaded df_test - shape: (1126, 

In [7]:
display(df_test)

,ID,Text,Label
0,Tu_OLI_01,Sabashanedu harate jasthi undu,blended hope
1,Tu_OLI_02,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ,uninvolved
2,Tu_OLI_03,Ayena pukuli Olu nadunu..,uninvolved
3,Tu_OLI_04,e sarti intro matrana,blended hope
4,Tu_OLI_05,#ಬೆಂಗಳೂರು 2024-25 ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊ...,uninvolved
...,...,...,...
1121,Tu_OLI_1122,sonte atthe bonte,uninvolved
1122,Tu_OLI_1123,2020 d yerla thuvondullara,uninvolved
1123,Tu_OLI_1124,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ... ಎಡ್ಡೆ ಸಂದೇಶ,encouraging hope
1124,Tu_OLI_1125,Vijay Anne thoode baaki aather😀,uninvolved


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re
import string
import emoji
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report,precision_score, recall_score, f1_score

In [9]:
df_train = df_train.dropna(subset=["Text"])
df_train["text"] = df_train["Text"].astype(str)
print(df_train['text'])

df_dev = df_dev.dropna(subset=["Text"])
df_dev["text"] = df_dev["Text"].astype(str)
print(df_dev['text'])

df_test  = df_test.dropna(subset=["Text"])
df_test["text"]  = df_test["Text"].astype(str)
print(df_test['text'])

0           Santhu nu masth miss malthondu ullar bro 🤔🤔🤔🤔
1        Undhe n YouTube d thooyere andala mobile bodathe
2       ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್‌ನ್ ನೀರ್‌ಡ್ ಪಾಡ್ದ್ ಅವು...
3                                 Super comedy deepak rai
4                     Namma tulundadu comedy first eppodu
                              ...                        
5986    Nanala bahubali n  kertinaye      tikdijena co...
5987    ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್‌ದ ಮುಂಗೆ. ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊ...
5988                                    Super undu teaser
5989                                    Deepak Anna super
5990                        Full Movie please Add malpule
Name: text, Length: 5991, dtype: object
0                           Yeranda narrd korle marre 😂😂😂
1       Tulu barpundge areg. Wow mokedha base tulu. Ja...
2                               Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale
3                             bayya poora film padlethe..
4       Super act super comedy deepak jp thumbinadu ra...
                              ..

In [10]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()
stemmer = PorterStemmer()

In [12]:
def preprocess(text):
    if not isinstance(text, str):
        return text
    # Remove URLs
    text =re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove emails
    text =re.sub(r'\S+@\S+', '', text)
    # Remove emojis
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Remove punctuations & numbers
    text=re.sub(r'\d+', '', text)
    # keeps english and kannada text
    text = re.sub(r'[^a-zA-Z\u0C80-\u0CFF\s]', '', text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenize using indic_tokenize
    tokens=indic_tokenize.trivial_tokenize(text, lang='kn')
    return ' '.join(tokens)

df_train['processed_text'] = df_train['text'].apply(preprocess)
df_dev['processed_text'] = df_dev['text'].apply(preprocess)
df_test['processed_text'] = df_test['text'].apply(preprocess)

display(df_train[['processed_text']])
display(df_dev[['processed_text']])
display(df_test[['processed_text']])

,processed_text
0,Santhu nu masth miss malthondu ullar bro think...
1,Undhe n YouTube d thooyere andala mobile bodathe
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...
3,Super comedy deepak rai
4,Namma tulundadu comedy first eppodu
...,...
5986,Nanala bahubali n kertinaye tikdijena comedy
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...
5988,Super undu teaser
5989,Deepak Anna super


,processed_text
0,Yeranda narrd korle marre facewithtearsofjoy f...
1,Tulu barpundge areg Wow mokedha base tulu Jai ...
2,Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale
3,bayya poora film padlethe
4,Super act super comedy deepak jp thumbinadu ra...
...,...
1279,Ancha Carecter wise thupinanda Police Carecter...
1280,Embna yenchina pukuli maya maraya
1281,Aredala patheryara panle
1282,da baala movie da seen da laka undu


,processed_text
0,Sabashanedu harate jasthi undu
1,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ
2,Ayena pukuli Olu nadunu
3,e sarti intro matrana
4,ಬೆಂಗಳೂರು ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊಕ ಎಗ್ಗದೆ ಕ...
...,...
1121,sonte atthe bonte
1122,d yerla thuvondullara
1123,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ ಸಂದೇಶ
1124,Vijay Anne thoode baaki aather grinningface


#**word ngrams**

In [13]:
def generate_word_ngrams_joined(tokens, n):
    if not isinstance(tokens, list):
        return []
    return [' '.join(gram) for gram in ngrams(tokens, n)]

# Step 1: Convert processed text into tokens (list of words)
df_train['tokens'] = df_train['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])
df_dev['tokens']  = df_dev['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])
df_test['tokens']  = df_test['processed_text'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Step 2: Generate word n-grams
for df in [df_train, df_dev, df_test]:
    df['word_unigrams_str'] = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 1))
    df['word_bigrams_str']  = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 2))
    df['word_trigrams_str'] = df['tokens'].apply(lambda tokens: generate_word_ngrams_joined(tokens, 3))

# Step 3: Display results
display(df_train[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])
display(df_dev[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])
display(df_test[['processed_text', 'word_unigrams_str', 'word_bigrams_str', 'word_trigrams_str']])

# ---- Merge unigrams, bigrams, trigrams into a single list ----
df_train['word_ngrams'] = (df_train['word_unigrams_str']+ df_train['word_bigrams_str']+ df_train['word_trigrams_str'])
df_dev['word_ngrams'] = (df_dev['word_unigrams_str']+ df_dev['word_bigrams_str']+ df_dev['word_trigrams_str'])
df_test['word_ngrams'] = (df_test['word_unigrams_str']+ df_test['word_bigrams_str']+ df_test['word_trigrams_str'])

display(df_train[['processed_text', 'word_ngrams']])

,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Santhu nu masth miss malthondu ullar bro think...,"[Santhu, nu, masth, miss, malthondu, ullar, br...","[Santhu nu, nu masth, masth miss, miss malthon...","[Santhu nu masth, nu masth miss, masth miss ma..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[Undhe, n, YouTube, d, thooyere, andala, mobil...","[Undhe n, n YouTube, YouTube d, d thooyere, th...","[Undhe n YouTube, n YouTube d, YouTube d thooy..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತಟ್ಟಿ, ನುಂಗೆಲ್, ತಾರೆದ, ಮಡಲ್ನ್, ನೀರ್ಡ್, ಪಾಡ್ದ್...","[ತಟ್ಟಿ ನುಂಗೆಲ್, ನುಂಗೆಲ್ ತಾರೆದ, ತಾರೆದ ಮಡಲ್ನ್, ಮ...","[ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ, ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್, ತಾ..."
3,Super comedy deepak rai,"[Super, comedy, deepak, rai]","[Super comedy, comedy deepak, deepak rai]","[Super comedy deepak, comedy deepak rai]"
4,Namma tulundadu comedy first eppodu,"[Namma, tulundadu, comedy, first, eppodu]","[Namma tulundadu, tulundadu comedy, comedy fir...","[Namma tulundadu comedy, tulundadu comedy firs..."
...,...,...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[Nanala, bahubali, n, kertinaye, tikdijena, co...","[Nanala bahubali, bahubali n, n kertinaye, ker...","[Nanala bahubali n, bahubali n kertinaye, n ke..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕಣಿಲೆ, ಪಂಡ, ಬೆದ್ರ್ದ, ಮುಂಗೆ, ಕನಿಲೆದ, ಉಪ್ಪಡ್, ಜ...","[ಕಣಿಲೆ ಪಂಡ, ಪಂಡ ಬೆದ್ರ್ದ, ಬೆದ್ರ್ದ ಮುಂಗೆ, ಮುಂಗೆ ...","[ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ, ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ, ಬೆದ್ರ್ದ..."
5988,Super undu teaser,"[Super, undu, teaser]","[Super undu, undu teaser]",[Super undu teaser]
5989,Deepak Anna super,"[Deepak, Anna, super]","[Deepak Anna, Anna super]",[Deepak Anna super]


,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Yeranda narrd korle marre facewithtearsofjoy f...,"[Yeranda, narrd, korle, marre, facewithtearsof...","[Yeranda narrd, narrd korle, korle marre, marr...","[Yeranda narrd korle, narrd korle marre, korle..."
1,Tulu barpundge areg Wow mokedha base tulu Jai ...,"[Tulu, barpundge, areg, Wow, mokedha, base, tu...","[Tulu barpundge, barpundge areg, areg Wow, Wow...","[Tulu barpundge areg, barpundge areg Wow, areg..."
2,Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale,"[Bಅರಿ, ಪೊರ್ಲು, ದ, ಬಲೆ, telpale]","[Bಅರಿ ಪೊರ್ಲು, ಪೊರ್ಲು ದ, ದ ಬಲೆ, ಬಲೆ telpale]","[Bಅರಿ ಪೊರ್ಲು ದ, ಪೊರ್ಲು ದ ಬಲೆ, ದ ಬಲೆ telpale]"
3,bayya poora film padlethe,"[bayya, poora, film, padlethe]","[bayya poora, poora film, film padlethe]","[bayya poora film, poora film padlethe]"
4,Super act super comedy deepak jp thumbinadu ra...,"[Super, act, super, comedy, deepak, jp, thumbi...","[Super act, act super, super comedy, comedy de...","[Super act super, act super comedy, super come..."
...,...,...,...,...
1279,Ancha Carecter wise thupinanda Police Carecter...,"[Ancha, Carecter, wise, thupinanda, Police, Ca...","[Ancha Carecter, Carecter wise, wise thupinand...","[Ancha Carecter wise, Carecter wise thupinanda..."
1280,Embna yenchina pukuli maya maraya,"[Embna, yenchina, pukuli, maya, maraya]","[Embna yenchina, yenchina pukuli, pukuli maya,...","[Embna yenchina pukuli, yenchina pukuli maya, ..."
1281,Aredala patheryara panle,"[Aredala, patheryara, panle]","[Aredala patheryara, patheryara panle]",[Aredala patheryara panle]
1282,da baala movie da seen da laka undu,"[da, baala, movie, da, seen, da, laka, undu]","[da baala, baala movie, movie da, da seen, see...","[da baala movie, baala movie da, movie da seen..."


,processed_text,word_unigrams_str,word_bigrams_str,word_trigrams_str
0,Sabashanedu harate jasthi undu,"[Sabashanedu, harate, jasthi, undu]","[Sabashanedu harate, harate jasthi, jasthi undu]","[Sabashanedu harate jasthi, harate jasthi undu]"
1,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ,"[ರಭಸೊಗು, ಸೋತುದು, ಇಂಚಾಯೆನೆ]","[ರಭಸೊಗು ಸೋತುದು, ಸೋತುದು ಇಂಚಾಯೆನೆ]",[ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ]
2,Ayena pukuli Olu nadunu,"[Ayena, pukuli, Olu, nadunu]","[Ayena pukuli, pukuli Olu, Olu nadunu]","[Ayena pukuli Olu, pukuli Olu nadunu]"
3,e sarti intro matrana,"[e, sarti, intro, matrana]","[e sarti, sarti intro, intro matrana]","[e sarti intro, sarti intro matrana]"
4,ಬೆಂಗಳೂರು ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊಕ ಎಗ್ಗದೆ ಕ...,"[ಬೆಂಗಳೂರು, ನೇ, ಸಾಲ್, ದ, ಪೊಸ, ಕಮಿಟಿ, ರಚನೆ, ಬೊಕ,...","[ಬೆಂಗಳೂರು ನೇ, ನೇ ಸಾಲ್, ಸಾಲ್ ದ, ದ ಪೊಸ, ಪೊಸ ಕಮಿಟ...","[ಬೆಂಗಳೂರು ನೇ ಸಾಲ್, ನೇ ಸಾಲ್ ದ, ಸಾಲ್ ದ ಪೊಸ, ದ ಪೊ..."
...,...,...,...,...
1121,sonte atthe bonte,"[sonte, atthe, bonte]","[sonte atthe, atthe bonte]",[sonte atthe bonte]
1122,d yerla thuvondullara,"[d, yerla, thuvondullara]","[d yerla, yerla thuvondullara]",[d yerla thuvondullara]
1123,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ ಸಂದೇಶ,"[ಬಾರಿ, ಪೊರ್ಲುದ, ಪ್ರಯತ್ನ, ಎಡ್ಡೆ, ಸಂದೇಶ]","[ಬಾರಿ ಪೊರ್ಲುದ, ಪೊರ್ಲುದ ಪ್ರಯತ್ನ, ಪ್ರಯತ್ನ ಎಡ್ಡೆ,...","[ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ, ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ, ..."
1124,Vijay Anne thoode baaki aather grinningface,"[Vijay, Anne, thoode, baaki, aather, grinningf...","[Vijay Anne, Anne thoode, thoode baaki, baaki ...","[Vijay Anne thoode, Anne thoode baaki, thoode ..."


,processed_text,word_ngrams
0,Santhu nu masth miss malthondu ullar bro think...,"[Santhu, nu, masth, miss, malthondu, ullar, br..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[Undhe, n, YouTube, d, thooyere, andala, mobil..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತಟ್ಟಿ, ನುಂಗೆಲ್, ತಾರೆದ, ಮಡಲ್ನ್, ನೀರ್ಡ್, ಪಾಡ್ದ್..."
3,Super comedy deepak rai,"[Super, comedy, deepak, rai, Super comedy, com..."
4,Namma tulundadu comedy first eppodu,"[Namma, tulundadu, comedy, first, eppodu, Namm..."
...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[Nanala, bahubali, n, kertinaye, tikdijena, co..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕಣಿಲೆ, ಪಂಡ, ಬೆದ್ರ್ದ, ಮುಂಗೆ, ಕನಿಲೆದ, ಉಪ್ಪಡ್, ಜ..."
5988,Super undu teaser,"[Super, undu, teaser, Super undu, undu teaser,..."
5989,Deepak Anna super,"[Deepak, Anna, super, Deepak Anna, Anna super,..."


#**char ngrams**

In [14]:
def text_graphemes(text):
    if not isinstance(text, str):
        return []
    # normalize spaces
    text = text.strip()
    return re.findall(r'\X', text)   # get extended grapheme clusters (works for all scripts)

def char_ngrams(text, n):
    tokens = text_graphemes(text)
    return [''.join(g) for g in ngrams(tokens, n)]

df_train['char_unigrams'] = df_train['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_train['char_bigrams']  = df_train['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_train['char_trigrams'] = df_train['processed_text'].apply(lambda x: char_ngrams(x, 3))

df_dev['char_unigrams'] = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_dev['char_bigrams']  = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_dev['char_trigrams'] = df_dev['processed_text'].apply(lambda x: char_ngrams(x, 3))

df_test['char_unigrams'] = df_test['processed_text'].apply(lambda x: char_ngrams(x, 1))
df_test['char_bigrams']  = df_test['processed_text'].apply(lambda x: char_ngrams(x, 2))
df_test['char_trigrams'] = df_test['processed_text'].apply(lambda x: char_ngrams(x, 3))

display(df_train[['processed_text','char_unigrams','char_bigrams','char_trigrams']])
display(df_dev[['processed_text','char_unigrams','char_bigrams','char_trigrams']])
display(df_test[['processed_text','char_unigrams','char_bigrams','char_trigrams']])

def normalize_ngram(ngram):
    if not isinstance(ngram, str):
        return ""
    return unicodedata.normalize("NFC", ngram)

def merge_ngram_columns(row):
    unigrams = [normalize_ngram(t) for t in row.get("char_unigrams", []) if isinstance(t, str)]
    bigrams  = [normalize_ngram(t) for t in row.get("char_bigrams", []) if isinstance(t, str)]
    trigrams = [normalize_ngram(t) for t in row.get("char_trigrams", []) if isinstance(t, str)]

    return unigrams + bigrams + trigrams

df_train["char_ngrams"] = df_train.apply(merge_ngram_columns, axis=1)
df_dev["char_ngrams"]  = df_dev.apply(merge_ngram_columns, axis=1)
df_test["char_ngrams"]  = df_test.apply(merge_ngram_columns, axis=1)

display(df_train["char_ngrams"])

,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Santhu nu masth miss malthondu ullar bro think...,"[S, a, n, t, h, u, , n, u, , m, a, s, t, h, ...","[Sa, an, nt, th, hu, u , n, nu, u , m, ma, a...","[San, ant, nth, thu, hu , u n, nu, nu , u m, ..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[U, n, d, h, e, , n, , Y, o, u, T, u, b, e, ...","[Un, nd, dh, he, e , n, n , Y, Yo, ou, uT, T...","[Und, ndh, dhe, he , e n, n , n Y, Yo, You, ..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತ, ಟ್, ಟಿ, , ನುಂ, ಗೆ, ಲ್, , ತಾ, ರೆ, ದ, , ಮ...","[ತಟ್, ಟ್ಟಿ, ಟಿ , ನುಂ, ನುಂಗೆ, ಗೆಲ್, ಲ್ , ತಾ, ...","[ತಟ್ಟಿ, ಟ್ಟಿ , ಟಿ ನುಂ, ನುಂಗೆ, ನುಂಗೆಲ್, ಗೆಲ್ ,..."
3,Super comedy deepak rai,"[S, u, p, e, r, , c, o, m, e, d, y, , d, e, ...","[Su, up, pe, er, r , c, co, om, me, ed, dy, y...","[Sup, upe, per, er , r c, co, com, ome, med, ..."
4,Namma tulundadu comedy first eppodu,"[N, a, m, m, a, , t, u, l, u, n, d, a, d, u, ...","[Na, am, mm, ma, a , t, tu, ul, lu, un, nd, d...","[Nam, amm, mma, ma , a t, tu, tul, ulu, lun, ..."
...,...,...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[N, a, n, a, l, a, , b, a, h, u, b, a, l, i, ...","[Na, an, na, al, la, a , b, ba, ah, hu, ub, b...","[Nan, ana, nal, ala, la , a b, ba, bah, ahu, ..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕ, ಣಿ, ಲೆ, , ಪಂ, ಡ, , ಬೆ, ದ್, ರ್, ದ, , ಮುಂ...","[ಕಣಿ, ಣಿಲೆ, ಲೆ , ಪಂ, ಪಂಡ, ಡ , ಬೆ, ಬೆದ್, ದ್ರ್...","[ಕಣಿಲೆ, ಣಿಲೆ , ಲೆ ಪಂ, ಪಂಡ, ಪಂಡ , ಡ ಬೆ, ಬೆದ್,..."
5988,Super undu teaser,"[S, u, p, e, r, , u, n, d, u, , t, e, a, s, ...","[Su, up, pe, er, r , u, un, nd, du, u , t, t...","[Sup, upe, per, er , r u, un, und, ndu, du , ..."
5989,Deepak Anna super,"[D, e, e, p, a, k, , A, n, n, a, , s, u, p, ...","[De, ee, ep, pa, ak, k , A, An, nn, na, a , ...","[Dee, eep, epa, pak, ak , k A, An, Ann, nna, ..."


,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Yeranda narrd korle marre facewithtearsofjoy f...,"[Y, e, r, a, n, d, a, , n, a, r, r, d, , k, ...","[Ye, er, ra, an, nd, da, a , n, na, ar, rr, r...","[Yer, era, ran, and, nda, da , a n, na, nar, ..."
1,Tulu barpundge areg Wow mokedha base tulu Jai ...,"[T, u, l, u, , b, a, r, p, u, n, d, g, e, , ...","[Tu, ul, lu, u , b, ba, ar, rp, pu, un, nd, d...","[Tul, ulu, lu , u b, ba, bar, arp, rpu, pun, ..."
2,Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale,"[B, ಅ, ರಿ, , ಪೊ, ರ್, ಲು, , ದ, , ಬ, ಲೆ, , t...","[Bಅ, ಅರಿ, ರಿ , ಪೊ, ಪೊರ್, ರ್ಲು, ಲು , ದ, ದ , ...","[Bಅರಿ, ಅರಿ , ರಿ ಪೊ, ಪೊರ್, ಪೊರ್ಲು, ರ್ಲು , ಲು ದ..."
3,bayya poora film padlethe,"[b, a, y, y, a, , p, o, o, r, a, , f, i, l, ...","[ba, ay, yy, ya, a , p, po, oo, or, ra, a , ...","[bay, ayy, yya, ya , a p, po, poo, oor, ora, ..."
4,Super act super comedy deepak jp thumbinadu ra...,"[S, u, p, e, r, , a, c, t, , s, u, p, e, r, ...","[Su, up, pe, er, r , a, ac, ct, t , s, su, u...","[Sup, upe, per, er , r a, ac, act, ct , t s, ..."
...,...,...,...,...
1279,Ancha Carecter wise thupinanda Police Carecter...,"[A, n, c, h, a, , C, a, r, e, c, t, e, r, , ...","[An, nc, ch, ha, a , C, Ca, ar, re, ec, ct, t...","[Anc, nch, cha, ha , a C, Ca, Car, are, rec, ..."
1280,Embna yenchina pukuli maya maraya,"[E, m, b, n, a, , y, e, n, c, h, i, n, a, , ...","[Em, mb, bn, na, a , y, ye, en, nc, ch, hi, i...","[Emb, mbn, bna, na , a y, ye, yen, enc, nch, ..."
1281,Aredala patheryara panle,"[A, r, e, d, a, l, a, , p, a, t, h, e, r, y, ...","[Ar, re, ed, da, al, la, a , p, pa, at, th, h...","[Are, red, eda, dal, ala, la , a p, pa, pat, ..."
1282,da baala movie da seen da laka undu,"[d, a, , b, a, a, l, a, , m, o, v, i, e, , ...","[da, a , b, ba, aa, al, la, a , m, mo, ov, v...","[da , a b, ba, baa, aal, ala, la , a m, mo, ..."


,processed_text,char_unigrams,char_bigrams,char_trigrams
0,Sabashanedu harate jasthi undu,"[S, a, b, a, s, h, a, n, e, d, u, , h, a, r, ...","[Sa, ab, ba, as, sh, ha, an, ne, ed, du, u , ...","[Sab, aba, bas, ash, sha, han, ane, ned, edu, ..."
1,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ,"[ರ, ಭ, ಸೊ, ಗು, , ಸೋ, ತು, ದು, , ಇಂ, ಚಾ, ಯೆ, ನೆ]","[ರಭ, ಭಸೊ, ಸೊಗು, ಗು , ಸೋ, ಸೋತು, ತುದು, ದು , ಇಂ...","[ರಭಸೊ, ಭಸೊಗು, ಸೊಗು , ಗು ಸೋ, ಸೋತು, ಸೋತುದು, ತುದ..."
2,Ayena pukuli Olu nadunu,"[A, y, e, n, a, , p, u, k, u, l, i, , O, l, ...","[Ay, ye, en, na, a , p, pu, uk, ku, ul, li, i...","[Aye, yen, ena, na , a p, pu, puk, uku, kul, ..."
3,e sarti intro matrana,"[e, , s, a, r, t, i, , i, n, t, r, o, , m, ...","[e , s, sa, ar, rt, ti, i , i, in, nt, tr, r...","[e s, sa, sar, art, rti, ti , i i, in, int, ..."
4,ಬೆಂಗಳೂರು ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊಕ ಎಗ್ಗದೆ ಕ...,"[ಬೆಂ, ಗ, ಳೂ, ರು, , ನೇ, , ಸಾ, ಲ್, , ದ, , ಪೊ...","[ಬೆಂಗ, ಗಳೂ, ಳೂರು, ರು , ನೇ, ನೇ , ಸಾ, ಸಾಲ್, ಲ್...","[ಬೆಂಗಳೂ, ಗಳೂರು, ಳೂರು , ರು ನೇ, ನೇ , ನೇ ಸಾ, ಸಾ..."
...,...,...,...,...
1121,sonte atthe bonte,"[s, o, n, t, e, , a, t, t, h, e, , b, o, n, ...","[so, on, nt, te, e , a, at, tt, th, he, e , ...","[son, ont, nte, te , e a, at, att, tth, the, ..."
1122,d yerla thuvondullara,"[d, , y, e, r, l, a, , t, h, u, v, o, n, d, ...","[d , y, ye, er, rl, la, a , t, th, hu, uv, v...","[d y, ye, yer, erl, rla, la , a t, th, thu, ..."
1123,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ ಸಂದೇಶ,"[ಬಾ, ರಿ, , ಪೊ, ರ್, ಲು, ದ, , ಪ್, ರ, ಯ, ತ್, ನ,...","[ಬಾರಿ, ರಿ , ಪೊ, ಪೊರ್, ರ್ಲು, ಲುದ, ದ , ಪ್, ಪ್ರ...","[ಬಾರಿ , ರಿ ಪೊ, ಪೊರ್, ಪೊರ್ಲು, ರ್ಲುದ, ಲುದ , ದ ಪ..."
1124,Vijay Anne thoode baaki aather grinningface,"[V, i, j, a, y, , A, n, n, e, , t, h, o, o, ...","[Vi, ij, ja, ay, y , A, An, nn, ne, e , t, t...","[Vij, ija, jay, ay , y A, An, Ann, nne, ne , ..."


,char_ngrams
0,"[S, a, n, t, h, u, , n, u, , m, a, s, t, h, ..."
1,"[U, n, d, h, e, , n, , Y, o, u, T, u, b, e, ..."
2,"[ತ, ಟ್, ಟಿ, , ನುಂ, ಗೆ, ಲ್, , ತಾ, ರೆ, ದ, , ಮ..."
3,"[S, u, p, e, r, , c, o, m, e, d, y, , d, e, ..."
4,"[N, a, m, m, a, , t, u, l, u, n, d, a, d, u, ..."
...,...
5986,"[N, a, n, a, l, a, , b, a, h, u, b, a, l, i, ..."
5987,"[ಕ, ಣಿ, ಲೆ, , ಪಂ, ಡ, , ಬೆ, ದ್, ರ್, ದ, , ಮುಂ..."
5988,"[S, u, p, e, r, , u, n, d, u, , t, e, a, s, ..."
5989,"[D, e, e, p, a, k, , A, n, n, a, , s, u, p, ..."


In [15]:
from indicsyllabifier.core import Syllabalizer
import re

syllabifier = Syllabalizer()

# very simple English syllable splitter (rule-based)
def english_syllables(word):
    word = word.lower()
    if not word.isalpha():
        return []
    # split based on vowel groups
    return re.findall(r'[^aeiouy]*[aeiouy]+(?:[^aeiouy]*)', word)

# detect Kannada characters
def contains_kannada(text):
    return bool(re.search(r'[\u0C80-\u0CFF]', text))

def get_syllables_bilingual(text):
    if not isinstance(text, str) or text.strip() == "":
      return []

    syllables = []
    tokens = text.split()
    for token in tokens:
      if contains_kannada(token):
        try:
          syls = syllabifier.syllabify(token)
          syllables.extend(syls)
        except:
          pass

      elif token.isascii():
        syls = english_syllables(token)
        syllables.extend(syls)
    return syllables

def syllable_ngrams(syllables, n):
    return [''.join(syllables[i:i+n]) for i in range(len(syllables) - n + 1)]


df_train['syllables'] = df_train['processed_text'].apply(get_syllables_bilingual)
df_train['syllable_bigrams'] = df_train['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_train['syllable_trigrams'] = df_train['syllables'].apply(lambda x: syllable_ngrams(x, 3))

df_dev['syllables'] = df_dev['processed_text'].apply(get_syllables_bilingual)
df_dev['syllable_bigrams'] = df_dev['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_dev['syllable_trigrams'] = df_dev['syllables'].apply(lambda x: syllable_ngrams(x, 3))

df_test['syllables'] = df_test['processed_text'].apply(get_syllables_bilingual)
df_test['syllable_bigrams'] = df_test['syllables'].apply(lambda x: syllable_ngrams(x, 2))
df_test['syllable_trigrams'] = df_test['syllables'].apply(lambda x: syllable_ngrams(x, 3))

display(df_train[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])
display(df_dev[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])
display(df_test[['processed_text', 'syllables', 'syllable_bigrams', 'syllable_trigrams']])


def normalize_syllable_ngram(ngram):
    if not isinstance(ngram, str):
        return ""
    return unicodedata.normalize("NFC", ngram)

def merge_syllable_ngram_columns(row):
    unigrams = [normalize_syllable_ngram(t) for t in row.get("syllables", []) if isinstance(t, str)]
    bigrams  = [normalize_syllable_ngram(t) for t in row.get("syllable_bigrams", []) if isinstance(t, str)]
    trigrams = [normalize_syllable_ngram(t) for t in row.get("syllable_trigrams", []) if isinstance(t, str)]
    return unigrams + bigrams + trigrams

df_train["syllable_ngrams"] = df_train.apply(merge_syllable_ngram_columns, axis=1)
df_dev["syllable_ngrams"]  = df_dev.apply(merge_syllable_ngram_columns, axis=1)
df_test["syllable_ngrams"]  = df_test.apply(merge_syllable_ngram_columns, axis=1)

display(df_train["syllable_ngrams"])

,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Santhu nu masth miss malthondu ullar bro think...,"[santh, u, nu, masth, miss, malth, ond, u, ull...","[santhu, unu, numasth, masthmiss, missmalth, m...","[santhunu, unumasth, numasthmiss, masthmissmal..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[undh, e, yout, ub, e, thooyer, e, and, al, a,...","[undhe, eyout, youtub, ube, ethooyer, thooyere...","[undheyout, eyoutub, youtube, ubethooyer, etho..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತ, ಟ್ಟಿ, ನುಂ, ಗೆ, ಲ್, ತಾ, ರೆ, ದ, ಮ, ಡ, ಲ್ನ್, ...","[ತಟ್ಟಿ, ಟ್ಟಿನುಂ, ನುಂಗೆ, ಗೆಲ್, ಲ್ತಾ, ತಾರೆ, ರೆದ,...","[ತಟ್ಟಿನುಂ, ಟ್ಟಿನುಂಗೆ, ನುಂಗೆಲ್, ಗೆಲ್ತಾ, ಲ್ತಾರೆ,..."
3,Super comedy deepak rai,"[sup, er, com, ed, y, deep, ak, rai]","[super, ercom, comed, edy, ydeep, deepak, akrai]","[supercom, ercomed, comedy, edydeep, ydeepak, ..."
4,Namma tulundadu comedy first eppodu,"[namm, a, tul, und, ad, u, com, ed, y, first, ...","[namma, atul, tulund, undad, adu, ucom, comed,...","[nammatul, atulund, tulundad, undadu, aducom, ..."
...,...,...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[nan, al, a, bah, ub, al, i, kert, in, aye, ti...","[nanal, ala, abah, bahub, ubal, ali, ikert, ke...","[nanala, alabah, abahub, bahubal, ubali, alike..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕ, ಣಿ, ಲೆ, ಪಂ, ಡ, ಬೆ, ದ್ರ್ದ, ಮುಂ, ಗೆ, ಕ, ನಿ, ...","[ಕಣಿ, ಣಿಲೆ, ಲೆಪಂ, ಪಂಡ, ಡಬೆ, ಬೆದ್ರ್ದ, ದ್ರ್ದಮುಂ,...","[ಕಣಿಲೆ, ಣಿಲೆಪಂ, ಲೆಪಂಡ, ಪಂಡಬೆ, ಡಬೆದ್ರ್ದ, ಬೆದ್ರ್..."
5988,Super undu teaser,"[sup, er, und, u, teas, er]","[super, erund, undu, uteas, teaser]","[superund, erundu, unduteas, uteaser]"
5989,Deepak Anna super,"[deep, ak, ann, a, sup, er]","[deepak, akann, anna, asup, super]","[deepakann, akanna, annasup, asuper]"


,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Yeranda narrd korle marre facewithtearsofjoy f...,"[yer, and, a, narrd, korl, e, marr, e, fac, ew...","[yerand, anda, anarrd, narrdkorl, korle, emarr...","[yeranda, andanarrd, anarrdkorl, narrdkorle, k..."
1,Tulu barpundge areg Wow mokedha base tulu Jai ...,"[tul, u, barp, undg, e, ar, eg, wow, mok, edh,...","[tulu, ubarp, barpundg, undge, ear, areg, egwo...","[tulubarp, ubarpundg, barpundge, undgear, eare..."
2,Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale,"[ , B, ಅ, ರ, ಿ, , ಪೊ, ರ್ಲು, ದ, ಬ, ಲೆ, telp, a...","[ B, Bಅ, ಅರ, ರಿ, ಿ , ಪೊ, ಪೊರ್ಲು, ರ್ಲುದ, ದಬ, ಬ...","[ Bಅ, Bಅರ, ಅರಿ, ರಿ , ಿ ಪೊ, ಪೊರ್ಲು, ಪೊರ್ಲುದ, ರ..."
3,bayya poora film padlethe,"[bayya, poor, a, film, padl, eth, e]","[bayyapoor, poora, afilm, filmpadl, padleth, e...","[bayyapoora, poorafilm, afilmpadl, filmpadleth..."
4,Super act super comedy deepak jp thumbinadu ra...,"[sup, er, act, sup, er, com, ed, y, deep, ak, ...","[super, eract, actsup, super, ercom, comed, ed...","[superact, eractsup, actsuper, supercom, ercom..."
...,...,...,...,...
1279,Ancha Carecter wise thupinanda Police Carecter...,"[anch, a, car, ect, er, wis, e, thup, in, and,...","[ancha, acar, carect, ecter, erwis, wise, ethu...","[anchacar, acarect, carecter, ecterwis, erwise..."
1280,Embna yenchina pukuli maya maraya,"[embn, a, yench, in, a, puk, ul, i, maya, mar,...","[embna, ayench, yenchin, ina, apuk, pukul, uli...","[embnayench, ayenchin, yenchina, inapuk, apuku..."
1281,Aredala patheryara panle,"[ar, ed, al, a, path, er, yar, a, panl, e]","[ared, edal, ala, apath, pather, eryar, yara, ...","[aredal, edala, alapath, apather, patheryar, e..."
1282,da baala movie da seen da laka undu,"[da, baal, a, mov, ie, da, seen, da, lak, a, u...","[dabaal, baala, amov, movie, ieda, daseen, see...","[dabaala, baalamov, amovie, movieda, iedaseen,..."


,processed_text,syllables,syllable_bigrams,syllable_trigrams
0,Sabashanedu harate jasthi undu,"[sab, ash, an, ed, u, har, at, e, jasth, i, un...","[sabash, ashan, aned, edu, uhar, harat, ate, e...","[sabashan, ashaned, anedu, eduhar, uharat, har..."
1,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ,"[ರ, ಭ, ಸೊ, ಗು, ಸೋ, ತು, ದು, ಇಂ, ಚಾ, ಯೆ, ನೆ]","[ರಭ, ಭಸೊ, ಸೊಗು, ಗುಸೋ, ಸೋತು, ತುದು, ದುಇಂ, ಇಂಚಾ, ...","[ರಭಸೊ, ಭಸೊಗು, ಸೊಗುಸೋ, ಗುಸೋತು, ಸೋತುದು, ತುದುಇಂ, ..."
2,Ayena pukuli Olu nadunu,"[ayen, a, puk, ul, i, ol, u, nad, un, u]","[ayena, apuk, pukul, uli, iol, olu, unad, nadu...","[ayenapuk, apukul, pukuli, uliol, iolu, olunad..."
3,e sarti intro matrana,"[e, sart, i, intr, o, matr, an, a]","[esart, sarti, iintr, intro, omatr, matran, ana]","[esarti, sartiintr, iintro, intromatr, omatran..."
4,ಬೆಂಗಳೂರು ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊಕ ಎಗ್ಗದೆ ಕ...,"[ಬೆಂ, ಗ, ಳೂ, ರು, ನೇ, ಸಾ, ಲ್, ದ, ಪೊ, ಸ, ಕ, ಮಿ, ...","[ಬೆಂಗ, ಗಳೂ, ಳೂರು, ರುನೇ, ನೇಸಾ, ಸಾಲ್, ಲ್ದ, ದಪೊ, ...","[ಬೆಂಗಳೂ, ಗಳೂರು, ಳೂರುನೇ, ರುನೇಸಾ, ನೇಸಾಲ್, ಸಾಲ್ದ,..."
...,...,...,...,...
1121,sonte atthe bonte,"[sont, e, atth, e, bont, e]","[sonte, eatth, atthe, ebont, bonte]","[sonteatth, eatthe, atthebont, ebonte]"
1122,d yerla thuvondullara,"[yerl, a, thuv, ond, ull, ar, a]","[yerla, athuv, thuvond, ondull, ullar, ara]","[yerlathuv, athuvond, thuvondull, ondullar, ul..."
1123,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ ಸಂದೇಶ,"[ಬಾ, ರಿ, ಪೊ, ರ್ಲು, ದ, ಪ್ರ, ಯ, ತ್ನ, ಎ, ಡ್ಡೆ, ಸಂ...","[ಬಾರಿ, ರಿಪೊ, ಪೊರ್ಲು, ರ್ಲುದ, ದಪ್ರ, ಪ್ರಯ, ಯತ್ನ, ...","[ಬಾರಿಪೊ, ರಿಪೊರ್ಲು, ಪೊರ್ಲುದ, ರ್ಲುದಪ್ರ, ದಪ್ರಯ, ಪ..."
1124,Vijay Anne thoode baaki aather grinningface,"[vij, ay, ann, e, thood, e, baak, i, aath, er,...","[vijay, ayann, anne, ethood, thoode, ebaak, ba...","[vijayann, ayanne, annethood, ethoode, thoodeb..."


,syllable_ngrams
0,"[santh, u, nu, masth, miss, malth, ond, u, ull..."
1,"[undh, e, yout, ub, e, thooyer, e, and, al, a,..."
2,"[ತ, ಟ್ಟಿ, ನುಂ, ಗೆ, ಲ್, ತಾ, ರೆ, ದ, ಮ, ಡ, ಲ್ನ್, ..."
3,"[sup, er, com, ed, y, deep, ak, rai, super, er..."
4,"[namm, a, tul, und, ad, u, com, ed, y, first, ..."
...,...
5986,"[nan, al, a, bah, ub, al, i, kert, in, aye, ti..."
5987,"[ಕ, ಣಿ, ಲೆ, ಪಂ, ಡ, ಬೆ, ದ್ರ್ದ, ಮುಂ, ಗೆ, ಕ, ನಿ, ..."
5988,"[sup, er, und, u, teas, er, super, erund, undu..."
5989,"[deep, ak, ann, a, sup, er, deepak, akann, ann..."


#**sub words**

In [16]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import re

def contains_kannada(text):
    return bool(re.search(r'[\u0C80-\u0CFF]', text))

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=12000,min_frequency=2,special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
tokenizer.train_from_iterator(df_train["processed_text"].astype(str).tolist(),trainer=trainer)

def extract_bilingual_bpe_tokens(text_series, tokenizer):
    return text_series.apply(lambda x: tokenizer.encode(x).tokens if isinstance(x, str) else [])

df_train["subwords"] = extract_bilingual_bpe_tokens(df_train["processed_text"], tokenizer)
df_dev["subwords"] = extract_bilingual_bpe_tokens(df_dev["processed_text"], tokenizer)
df_test["subwords"] = extract_bilingual_bpe_tokens(df_test["processed_text"], tokenizer)

df_train["subword_text"] = df_train["subwords"].apply(lambda x: " ".join(x))
df_dev["subword_text"]  = df_dev["subwords"].apply(lambda x: " ".join(x))
df_test["subword_text"]  = df_test["subwords"].apply(lambda x: " ".join(x))

display(df_train[["processed_text", "subwords"]])
display(df_dev[["processed_text", "subwords"]])
display(df_test[["processed_text", "subwords"]])

,processed_text,subwords
0,Santhu nu masth miss malthondu ullar bro think...,"[Santhu, nu, masth, miss, malth, ondu, ullar, ..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[Undhe, n, YouTube, d, thooyere, andala, mobil..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತ, ಟ್ಟಿ, ನು, ಂ, ಗೆಲ್, ತಾ, ರೆದ, ಮ, ಡ, ಲ್ನ್, ನೀ..."
3,Super comedy deepak rai,"[Super, comedy, deepak, rai]"
4,Namma tulundadu comedy first eppodu,"[Namma, tulu, nda, du, comedy, first, eppodu]"
...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[Nanala, bahubali, n, ker, tina, ye, tik, di, ..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕ, ಣಿ, ಲೆ, ಪಂಡ, ಬೆದ್, ರ್ದ, ಮು, ಂ, ಗೆ, ಕ, ನಿ, ..."
5988,Super undu teaser,"[Super, undu, te, aser]"
5989,Deepak Anna super,"[Deepak, Anna, super]"


,processed_text,subwords
0,Yeranda narrd korle marre facewithtearsofjoy f...,"[Yer, anda, nar, rd, korle, marre, facewithtea..."
1,Tulu barpundge areg Wow mokedha base tulu Jai ...,"[Tulu, barpund, ge, areg, Wow, mo, ke, dha, ba..."
2,Bಅರಿ ಪೊರ್ಲು ದ ಬಲೆ telpale,"[B, ಅರಿ, ಪೊರ್ಲು, ದ, ಬಲೆ, tel, pale]"
3,bayya poora film padlethe,"[bay, ya, poora, film, padle, the]"
4,Super act super comedy deepak jp thumbinadu ra...,"[Super, act, super, comedy, deepak, jp, thumb,..."
...,...,...
1279,Ancha Carecter wise thupinanda Police Carecter...,"[Ancha, C, are, cter, w, ise, thup, in, anda, ..."
1280,Embna yenchina pukuli maya maraya,"[E, mb, na, yenchina, pukuli, maya, maraya]"
1281,Aredala patheryara panle,"[Are, dala, patheryara, panle]"
1282,da baala movie da seen da laka undu,"[da, ba, ala, movie, da, seen, da, laka, undu]"


,processed_text,subwords
0,Sabashanedu harate jasthi undu,"[Sa, ba, shane, du, hara, te, jasthi, undu]"
1,ರಭಸೊಗು ಸೋತುದು ಇಂಚಾಯೆನೆ,"[ರ, ಭ, ಸೊಗು, ಸೋ, ತುದು, ಇಂಚ, ಾಯೆ, ನೆ]"
2,Ayena pukuli Olu nadunu,"[Ayena, pukuli, Olu, nadunu]"
3,e sarti intro matrana,"[e, sar, ti, intro, matrana]"
4,ಬೆಂಗಳೂರು ನೇ ಸಾಲ್ ದ ಪೊಸ ಕಮಿಟಿ ರಚನೆ ಬೊಕ ಎಗ್ಗದೆ ಕ...,"[ಬೆಂಗಳೂರು, ನೇ, ಸಾಲ್, ದ, ಪೊಸ, ಕ, ಮಿ, ಟಿ, ರ, ಚನೆ..."
...,...,...
1121,sonte atthe bonte,"[son, te, at, the, b, onte]"
1122,d yerla thuvondullara,"[d, yerla, thuv, ond, ullara]"
1123,ಬಾರಿ ಪೊರ್ಲುದ ಪ್ರಯತ್ನ ಎಡ್ಡೆ ಸಂದೇಶ,"[ಬಾರಿ, ಪೊರ್ಲುದ, ಪ್ರಯತ್ನ, ಎಡ್ಡೆ, ಸಂದೇಶ]"
1124,Vijay Anne thoode baaki aather grinningface,"[Vijay, Anne, thoo, de, baaki, aath, er, grinn..."


In [17]:
def merge_all_features(row):

    word_feats = row.get("word_ngrams", [])
    char_feats = row.get("char_ngrams", [])
    syll_feats = row.get("syllable_ngrams", [])
    subword_feats = row.get("subwords", [])

    merged = []

    if isinstance(word_feats, list):
        merged.extend(word_feats)

    if isinstance(char_feats, list):
        merged.extend(char_feats)

    if isinstance(syll_feats, list):
        merged.extend(syll_feats)

    if isinstance(subword_feats, list):
        merged.extend(subword_feats)

    return merged

df_train["all_features"] = df_train.apply(merge_all_features, axis=1)
df_dev["all_features"]  = df_dev.apply(merge_all_features, axis=1)
df_test["all_features"]  = df_test.apply(merge_all_features, axis=1)


display(df_train[["processed_text", "all_features"]])

,processed_text,all_features
0,Santhu nu masth miss malthondu ullar bro think...,"[Santhu, nu, masth, miss, malthondu, ullar, br..."
1,Undhe n YouTube d thooyere andala mobile bodathe,"[Undhe, n, YouTube, d, thooyere, andala, mobil..."
2,ತಟ್ಟಿ ನುಂಗೆಲ್ ತಾರೆದ ಮಡಲ್ನ್ ನೀರ್ಡ್ ಪಾಡ್ದ್ ಅವು ಬ...,"[ತಟ್ಟಿ, ನುಂಗೆಲ್, ತಾರೆದ, ಮಡಲ್ನ್, ನೀರ್ಡ್, ಪಾಡ್ದ್..."
3,Super comedy deepak rai,"[Super, comedy, deepak, rai, Super comedy, com..."
4,Namma tulundadu comedy first eppodu,"[Namma, tulundadu, comedy, first, eppodu, Namm..."
...,...,...
5986,Nanala bahubali n kertinaye tikdijena comedy,"[Nanala, bahubali, n, kertinaye, tikdijena, co..."
5987,ಕಣಿಲೆ ಪಂಡ ಬೆದ್ರ್ದ ಮುಂಗೆ ಕನಿಲೆದ ಉಪ್ಪಡ್ ಜೀರ್ಣೊಗು...,"[ಕಣಿಲೆ, ಪಂಡ, ಬೆದ್ರ್ದ, ಮುಂಗೆ, ಕನಿಲೆದ, ಉಪ್ಪಡ್, ಜ..."
5988,Super undu teaser,"[Super, undu, teaser, Super undu, undu teaser,..."
5989,Deepak Anna super,"[Deepak, Anna, super, Deepak Anna, Anna super,..."


In [18]:
df_train["all_features_text"] = df_train["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df_dev["all_features_text"] = df_dev["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df_test["all_features_text"] = df_test["all_features"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train["all_features_text"])
X_dev_tfidf = tfidf_vectorizer.transform(df_dev["all_features_text"])
X_test_tfidf = tfidf_vectorizer.transform(df_test["all_features_text"])

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total TF-IDF features:", len(feature_names))
print("First 50 features:", feature_names[:50])


Total TF-IDF features: 71752
First 50 features: ['_a' '_a_' '_aಎ' 'a_' 'a_a' 'aa' 'aaa' 'aaaa' 'aaaaa' 'aaaaaa' 'aaaaaaa'
 'aaaaaaaa' 'aaaaaaaafac' 'aaaaaaaafacew' 'aaaaaaapppp' 'aaaaaaappppeeeee'
 'aaaaaaasakk' 'aaaaaaasakkath' 'aaaaaaasanth' 'aaaaaaasanthuuu'
 'aaaaaandh' 'aaaaaandhmarr' 'aaaaar' 'aaaaareall' 'aaaaareally' 'aaaadad'
 'aaaadada' 'aaaae' 'aaaaeayen' 'aaaal' 'aaaalu' 'aaaam' 'aaaaman'
 'aaaamanag' 'aaaanam' 'aaaand' 'aaaandthudh' 'aaaarakk' 'aaaarakkil'
 'aaaayan' 'aaaayana' 'aaaayanapuk' 'aaaayench' 'aaaayenchin' 'aaabaah'
 'aaabaahub' 'aaac' 'aaacut' 'aaacute' 'aaad']


In [19]:
tfidf_df_train = pd.DataFrame(X_train_tfidf.toarray(),columns=feature_names)
tfidf_df_dev = pd.DataFrame(X_dev_tfidf.toarray(),columns=feature_names)
tfidf_df_test = pd.DataFrame(X_test_tfidf.toarray(),columns=feature_names)


display(tfidf_df_train)
display(tfidf_df_dev)
display(tfidf_df_test)

,_a,_a_,_aಎ,a_,a_a,aa,aaa,aaaa,aaaaa,aaaaaa,...,ಹನಪ,ಹನಸ,ಹಪ,ಹಬ,ಹರ,ಹಲ,ಹಳ,ಹಹ,ೠtul,ೠtulub
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5986,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5987,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5988,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5989,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,_a,_a_,_aಎ,a_,a_a,aa,aaa,aaaa,aaaaa,aaaaaa,...,ಹನಪ,ಹನಸ,ಹಪ,ಹಬ,ಹರ,ಹಲ,ಹಳ,ಹಹ,ೠtul,ೠtulub
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1279,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1280,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1281,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1282,0.0,0.0,0.0,0.0,0.0,0.021177,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,_a,_a_,_aಎ,a_,a_a,aa,aaa,aaaa,aaaaa,aaaaaa,...,ಹನಪ,ಹನಸ,ಹಪ,ಹಬ,ಹರ,ಹಲ,ಹಳ,ಹಹ,ೠtul,ೠtulub
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1121,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1122,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1123,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1124,0.0,0.0,0.0,0.0,0.0,0.059479,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# Voting Classifier
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

estimators = [(name, model) for name, model in models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting='soft')

# Train
voting_clf.fit(X_train_tfidf, df_train['Label'])

# -------------------------
# DEV Evaluation
# -------------------------
y_dev_pred = voting_clf.predict(X_dev_tfidf)

print("DEV RESULTS")
print("Accuracy:", accuracy_score(df_dev['Label'], y_dev_pred))
print("Macro F1:", f1_score(df_dev['Label'], y_dev_pred, average='macro'))
print(classification_report(df_dev['Label'], y_dev_pred, zero_division=0))

# -------------------------
# TEST Evaluation
# -------------------------
y_test_pred = voting_clf.predict(X_test_tfidf)

print("\nTEST RESULTS")
print("Accuracy:", accuracy_score(df_test['Label'], y_test_pred))
print("Macro F1:", f1_score(df_test['Label'], y_test_pred, average='macro'))
print(classification_report(df_test['Label'], y_test_pred, zero_division=0))

DEV RESULTS
Accuracy: 0.6596573208722741
Macro F1: 0.41658978433457516
                   precision    recall  f1-score   support

     blended hope       0.64      0.04      0.07       191
discouraging hope       0.88      0.05      0.09       153
 encouraging hope       0.70      0.81      0.75       406
       uninvolved       0.63      0.94      0.76       534

         accuracy                           0.66      1284
        macro avg       0.71      0.46      0.42      1284
     weighted avg       0.68      0.66      0.57      1284


TEST RESULTS
Accuracy: 0.6509769094138543
Macro F1: 0.4295821312700656
                   precision    recall  f1-score   support

     blended hope       0.75      0.07      0.13       172
discouraging hope       1.00      0.05      0.09       149
 encouraging hope       0.73      0.83      0.77       407
       uninvolved       0.59      0.95      0.73       398

         accuracy                           0.65      1126
        macro avg       0.